## objective of this notebook is to estimate the coverage of specific buildings in downtown seattle
- use tables created in notebook 01_Ingest and available in cmegdemos_catalog.network_analytics_enablement schema
- Perform spatial joins with ST functions to determine which buildings fall within specific coverage hexagons
- calculate distance between buildings and cell towers
- do analysis on distance to a cell tower and signal strength of the h3 grid that the building falls in
- visualize with a heatmap of coverage for downtown seattle - make sure plot is scrollable/zoomable
- use databricks native ST functions and H3 indexing after converting data appropriately wherever possible
- ultimately we should save another table that contains all of this data for buildings in downtown seattle with the h3 coverage information and distance from nearest tower

In [0]:
%pip install folium --quiet

In [0]:
CATALOG = "cmegdemos_catalog"
SCHEMA = "network_analytics_enablement"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# Downtown Seattle core bounding box
DOWNTOWN = {
    "lat_min": 47.595, "lat_max": 47.625,
    "lon_min": -122.355, "lon_max": -122.325
}

for t in ["fcc_bdc_h3_seattle", "building_footprints", "cell_towers"]:
    print(f"{t}: {spark.table(t).count():,} rows")

In [0]:
# Filter to downtown Seattle buildings, H3-index their centroids,
# and join with aggregated FCC BDC 5G coverage per H3 hex

df_bldg_coverage = spark.sql(f"""
    WITH bdc_per_hex AS (
        -- Aggregate BDC coverage per H3 hex (best available speeds)
        SELECT 
            h3_res9_id,
            MAX(mindown) AS best_download_mbps,
            MAX(minup) AS best_upload_mbps,
            COUNT(*) AS provider_count
        FROM fcc_bdc_h3_seattle
        GROUP BY h3_res9_id
    ),
    downtown_bldg AS (
        -- Filter buildings to downtown and compute centroids
        SELECT 
            building_id,
            height AS building_height,
            geometry,
            ST_Centroid(geometry) AS centroid,
            ST_X(ST_Centroid(geometry)) AS centroid_lon,
            ST_Y(ST_Centroid(geometry)) AS centroid_lat
        FROM building_footprints
        WHERE ST_Y(ST_Centroid(geometry)) BETWEEN {DOWNTOWN['lat_min']} AND {DOWNTOWN['lat_max']}
          AND ST_X(ST_Centroid(geometry)) BETWEEN {DOWNTOWN['lon_min']} AND {DOWNTOWN['lon_max']}
    ),
    bldg_h3 AS (
        -- Assign H3 resolution-9 index to each building centroid
        SELECT *,
            h3_h3tostring(h3_longlatash3(centroid_lon, centroid_lat, 9)) AS h3_res9_id
        FROM downtown_bldg
    )
    -- Join buildings with aggregated coverage
    SELECT 
        b.building_id, b.building_height, b.geometry, b.centroid,
        b.centroid_lon, b.centroid_lat, b.h3_res9_id,
        COALESCE(c.best_download_mbps, 0) AS best_download_mbps,
        COALESCE(c.best_upload_mbps, 0) AS best_upload_mbps,
        COALESCE(c.provider_count, 0) AS provider_count
    FROM bldg_h3 b
    LEFT JOIN bdc_per_hex c ON b.h3_res9_id = c.h3_res9_id
""")

df_bldg_coverage.createOrReplaceTempView("downtown_coverage")
total = df_bldg_coverage.count()
covered = df_bldg_coverage.filter("best_download_mbps > 0").count()
print(f"Downtown buildings: {total:,} | With 5G coverage: {covered:,} ({covered/total*100:.1f}%)")
display(df_bldg_coverage.select(
    "building_id", "building_height", "centroid_lon", "centroid_lat",
    "h3_res9_id", "best_download_mbps", "best_upload_mbps", "provider_count"
).limit(10))

In [0]:
# Find nearest cell tower for each downtown building using ST_DistanceSphere

df_nearest = spark.sql(f"""
    WITH nearby_towers AS (
        -- Pre-filter towers to area around downtown for efficiency
        SELECT tower_id, carrier, tower_type, cell_id,
               coverage_radius_m, samples, location
        FROM cell_towers
        WHERE latitude BETWEEN {DOWNTOWN['lat_min'] - 0.03} AND {DOWNTOWN['lat_max'] + 0.03}
          AND longitude BETWEEN {DOWNTOWN['lon_min'] - 0.03} AND {DOWNTOWN['lon_max'] + 0.03}
    ),
    distances AS (
        SELECT 
            b.building_id,
            t.tower_id AS nearest_tower_id,
            t.carrier AS nearest_carrier,
            t.tower_type AS nearest_tower_type,
            t.cell_id AS nearest_cell_id,
            t.coverage_radius_m AS nearest_coverage_radius_m,
            t.samples AS nearest_tower_samples,
            ROUND(ST_DistanceSphere(b.centroid, t.location), 1) AS distance_to_tower_m
        FROM downtown_coverage b
        CROSS JOIN nearby_towers t
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY building_id ORDER BY distance_to_tower_m) AS rn
        FROM distances
    )
    SELECT * EXCEPT(rn) FROM ranked WHERE rn = 1
""")

df_nearest.createOrReplaceTempView("nearest_tower")
print(f"Nearest tower computed for {df_nearest.count():,} buildings")
display(df_nearest.limit(10))

In [0]:
# Combine coverage + nearest tower into full analysis view
spark.sql("""
    CREATE OR REPLACE TEMP VIEW buildings_analysis AS
    SELECT 
        c.building_id, c.building_height, c.geometry,
        c.centroid_lon, c.centroid_lat, c.h3_res9_id,
        c.best_download_mbps, c.best_upload_mbps, c.provider_count,
        n.nearest_tower_id, n.nearest_carrier, n.nearest_tower_type,
        n.nearest_cell_id, n.nearest_coverage_radius_m,
        n.nearest_tower_samples, n.distance_to_tower_m
    FROM downtown_coverage c
    JOIN nearest_tower n ON c.building_id = n.building_id
""")

# Analyze relationship between distance to tower and coverage quality
print("Signal Strength vs Distance to Nearest Tower:")
df_stats = spark.sql("""
    SELECT 
        CASE 
            WHEN distance_to_tower_m < 200 THEN '1: < 200m'
            WHEN distance_to_tower_m < 500 THEN '2: 200-500m'
            WHEN distance_to_tower_m < 1000 THEN '3: 500m-1km'
            WHEN distance_to_tower_m < 2000 THEN '4: 1-2km'
            ELSE '5: > 2km'
        END AS distance_bucket,
        COUNT(*) AS buildings,
        ROUND(AVG(best_download_mbps), 1) AS avg_download_mbps,
        ROUND(AVG(best_upload_mbps), 1) AS avg_upload_mbps,
        ROUND(AVG(distance_to_tower_m), 0) AS avg_distance_m,
        ROUND(AVG(provider_count), 1) AS avg_providers
    FROM buildings_analysis
    WHERE best_download_mbps > 0
    GROUP BY 1
    ORDER BY 1
""")
display(df_stats)

In [0]:
import folium
from folium.plugins import HeatMap

# Collect building coverage data
pdf = spark.sql("""
    SELECT centroid_lat, centroid_lon, best_download_mbps, distance_to_tower_m
    FROM buildings_analysis
    WHERE best_download_mbps > 0
""").toPandas()

# Collect nearby tower locations (real OpenCellID data)
pdf_towers = spark.sql(f"""
    SELECT latitude, longitude, carrier, tower_type, cell_id, coverage_radius_m, samples
    FROM cell_towers
    WHERE latitude BETWEEN {DOWNTOWN['lat_min'] - 0.01} AND {DOWNTOWN['lat_max'] + 0.01}
      AND longitude BETWEEN {DOWNTOWN['lon_min'] - 0.01} AND {DOWNTOWN['lon_max'] + 0.01}
""").toPandas()

print(f"Plotting {len(pdf):,} buildings + {len(pdf_towers):,} T-Mobile towers")

# Create scrollable / zoomable map centered on downtown Seattle
m = folium.Map(location=[47.61, -122.34], zoom_start=15, tiles='CartoDB positron')

# Heatmap layer weighted by download speed
heat_data = pdf[['centroid_lat', 'centroid_lon', 'best_download_mbps']].values.tolist()
HeatMap(
    heat_data, radius=12, blur=8, max_zoom=18,
    gradient={0.2: 'blue', 0.4: 'cyan', 0.6: 'lime', 0.8: 'yellow', 1.0: 'red'}
).add_to(m)

# Cell tower markers (color-coded by radio type)
radio_colors = {'LTE': 'red', 'NR': 'black', 'GSM': 'orange', 'UMTS': 'gray'}
for _, row in pdf_towers.iterrows():
    color = radio_colors.get(row['tower_type'], 'blue')
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6, color=color, fill=True, fill_color=color, fill_opacity=0.85,
        popup=(
            f"<b>T-Mobile {row['tower_type']}</b><br>"
            f"Cell ID: {row['cell_id']}<br>"
            f"Range: {row['coverage_radius_m']}m<br>"
            f"Samples: {row['samples']}"
        )
    ).add_to(m)

# Legend
legend_html = """
<div style="
    position: fixed; 
    bottom: 30px; left: 30px; width: 200px;
    background-color: white; border:2px solid grey; z-index:9999; font-size:13px;
    padding: 10px; border-radius: 5px;">
    <b>Coverage Heatmap</b><br>
    <span style="background:blue;width:14px;height:14px;display:inline-block;"></span> Low &nbsp;
    <span style="background:lime;width:14px;height:14px;display:inline-block;"></span> Med &nbsp;
    <span style="background:red;width:14px;height:14px;display:inline-block;"></span> High<br>
    <hr style="margin:4px 0;">
    <b>Tower Type</b><br>
    <span style="background:red;width:14px;height:14px;display:inline-block;border-radius:50%;"></span> LTE &nbsp;
    <span style="background:black;width:14px;height:14px;display:inline-block;border-radius:50%;"></span> NR (5G) &nbsp;
    <span style="background:orange;width:14px;height:14px;display:inline-block;border-radius:50%;"></span> GSM
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

displayHTML(m._repr_html_())

In [0]:
# Persist the full analysis table for downstream use
table_name = f"{CATALOG}.{SCHEMA}.downtown_seattle_building_coverage"

spark.sql(f"""
    CREATE OR REPLACE TABLE {table_name} AS
    SELECT 
        building_id, building_height, geometry,
        centroid_lon, centroid_lat, h3_res9_id,
        best_download_mbps, best_upload_mbps, provider_count,
        nearest_tower_id, nearest_carrier, nearest_tower_type,
        nearest_cell_id, nearest_coverage_radius_m,
        nearest_tower_samples, distance_to_tower_m
    FROM buildings_analysis
""")

count = spark.table(table_name).count()
print(f"\u2713 Saved {count:,} rows to {table_name}")
print(f"  Columns: {', '.join(spark.table(table_name).columns)}")

In [0]:
# need to train a model to predict coverage relative to tower distance, may need to get distance from a real tower dataset to have some signal

In [0]:
# or just genie space